# Lab 01: Pandas, Data Cleaning, and Exploratory Data Analysis
**Week 2**

Your club ran a few campus events and collected feedback. Before deciding what to
host next, you need to make sense of the survey responses. Let's turn a messy table
into something we can actually learn from!

By the end, you'll be able to:
- Load and inspect a pandas DataFrame.
- Select, filter, sort, and add columns.
- Find duplicates, inconsistent labels, missing values, and invalid values.
- Use summary statistics and grouped summaries to answer questions about data.

Work from top to bottom. Run code cells with **Shift + Enter**. Replace each `...`
with your code and fill in the written responses. You can work with a neighbor and ask a tutor/TA for help.

## 1. Meet your DataFrame

**Pandas** is a Python library for working with tabular data. We import it as `pd`.
A **DataFrame** is a table with labeled rows and columns; a **Series** is one labeled
column. For example, `responses["event"]` returns a Series, while
`responses[["event", "rating"]]` returns a DataFrame.

CSV files store tables as comma-separated values. The survey is bundled as
`club_event_survey.csv` in the same folder as this notebook. Load it with
`pd.read_csv("club_event_survey.csv")`, then explore it using pandas.
**Run the setup below.**

| Column | Meaning |
|---|---|
| `student_id` | Fictional respondent ID; each student should appear once |
| `event` | The event the student attended |
| `hours_spent` | Reported time at that event, in hours |
| `rating` | Satisfaction rating, from 1 (lowest) to 5 (highest) |


In [ ]:
import pandas as pd
import otter

grader = otter.Notebook("Lab01.ipynb", tests_dir="tests/lab01")
responses = pd.read_csv("club_event_survey.csv")
print("Survey loaded! Use pandas below to explore it.")

### Q1. What's in the table?
Run the next two cells. `head()` previews rows; `shape` gives `(rows, columns)`;
`info()` reports column types and non-missing counts.

Notice that `shape` is an **attribute**, so it has no parentheses. Methods such as
`head()` and `info()` do use parentheses.


In [ ]:
responses.head()

In [ ]:
print("Rows and columns:", responses.shape)
responses.info()

**Your response:** How many rows and columns are there? Which two columns have
missing entries? Is every column numeric?

> Write your answer here.


## 2. Select, filter, and sort

Selecting columns lets us focus on relevant variables. Filtering keeps rows that
meet a condition. This complete example selects the ID and hours for visits shorter
than two hours:


In [ ]:
responses.loc[responses["hours_spent"] < 2, ["student_id", "hours_spent"]]

`.loc[rows, columns]` selects using labels or a Boolean condition. `.iloc[]` selects
by integer position: `responses.iloc[0]` gets the first row. Row labels and positions
can differ after sorting or removing rows.

### Q2. Find the longer visits
Select rows where `hours_spent` is **at least 2**, keeping only `student_id`, `event`,
and `hours_spent`. Then sort your result from most hours to fewest.

**Hint:** Use `>=` in the condition and
`.sort_values("hours_spent", ascending=False)` to sort.


In [ ]:
long_visits = responses.loc[..., ["student_id", "event", "hours_spent"]]
long_visits = ...
long_visits

In [ ]:
grader.check("q2")

## 3. Investigate the mess

**Exploratory data analysis (EDA)** means asking questions, inspecting data quality,
and looking for patterns before drawing conclusions. It starts *before* cleaning
and continues afterward.

### Q3. Audit the raw data
Run this audit. `isna().sum()` counts missing entries per column;
`duplicated().sum()` counts exact repeated rows after their first appearance;
`value_counts()` counts the occurrences of each value.


In [ ]:
print("Missing entries:")
print(responses.isna().sum())
print("\nExtra duplicate rows:", responses.duplicated().sum())
print("\nEvent labels:")
print(responses["event"].value_counts())
print("\nNumeric summaries:")
responses.describe()

**Your response:** Besides missing data, identify three data quality problems you
can see. Use the event labels, duplicate count, and allowed rating range as clues.

> Write your answer here.


## 4. Clean labels and duplicates

Let's preserve `responses` as our raw data and work on a copy. Cleaning decisions
should be deliberate: a repeated row would count one student's feedback twice,
and `Game Night` and `GAME NIGHT` should belong to the same event.

### Q4. Make event names consistent and remove the repeated response
1. Standardize `event` using `.str.strip().str.lower()`. This removes surrounding
   spaces and converts text to lowercase.
2. Remove exact duplicate rows using `.drop_duplicates()`.

Both methods return a result, so **assign the result** to keep the change.
Here we know the extra row is an accidental repeat. Matching IDs with conflicting
answers would require investigation instead of blindly keeping the first row.


In [ ]:
clean = responses.copy()
clean["event"] = ...
clean = ...

print("Rows remaining:", len(clean))
clean["event"].value_counts()

In [ ]:
grader.check("q4")

**Checkpoint:** You should have **11 rows** and **3 event categories**. If not,
revisit your cleaning code before continuing.


## 5. Handle invalid and missing values

A rating of 9 is **invalid** on a 1–5 scale. An unusually high but valid rating is
not automatically a mistake. We should use what the variable means to make this decision.

### Q5. Mark invalid ratings as missing
Use the mask below to replace out-of-range ratings with `float("nan")`, a numeric
missing value. The operator `|` means **or**; put each comparison in parentheses.

**Hint:** To update selected cells, use `clean.loc[row_condition, "column"] = value`.


In [ ]:
invalid_rating = (clean["rating"] < 1) | (clean["rating"] > 5)
clean.loc[invalid_rating, "rating"] = ...
clean["rating"]

In [ ]:
grader.check("q5")

### Q6. Choose how to handle missing data
There is no universal rule to drop or fill every missing entry.

For this exercise, **keep missing ratings missing**: we don't know those students'
opinions. Pandas means skip missing values, so later we'll also count how many
ratings each mean is based on.

For `hours_spent`, practice filling the one missing value with the **median** of
known hours. This is an assumption for the exercise, not a recovered observation.
We save a flag first so we can still identify which value was filled.

Complete the two blanks. Use `.median()` and `.fillna(value)`.


In [ ]:
clean["hours_were_missing"] = clean["hours_spent"].isna()
median_hours = ...
clean["hours_spent"] = clean["hours_spent"].fillna(...)

print("Missing entries after cleaning:")
print(clean.isna().sum())
clean

In [ ]:
grader.check("q6")

**Checkpoint:** `hours_spent` should have no missing values; `rating` should have
**2 missing values**. Missing values can be an honest part of a cleaned dataset.

**Your response:** Why might filling missing ratings with 5 make our conclusions
misleading?

> Write your answer here.


## 6. Manipulate the cleaned data

### Q7. Add a column and combine conditions
Create a Boolean column called `long_visit` that is `True` when `hours_spent` is
at least 2. Then select responses where `long_visit` is `True` **and** `rating`
is at least 4. Keep the ID, event, hours, and rating columns.

**Hint:** Combine conditions with `&`, not Python's `and`:
`(condition_one) & (condition_two)`. A comparison with a missing numeric rating
will not select that row.


In [ ]:
clean["long_visit"] = ...
happy_long_visits = clean.loc[
    ...,
    ["student_id", "event", "hours_spent", "rating"]
]
happy_long_visits

In [ ]:
grader.check("q7")

## 7. Explore and interpret

Now that the data is more consistent, let's return to our question: **What does the
feedback suggest about the events?** Run these summaries of the cleaned data.


In [ ]:
print("Responses per event:")
print(clean["event"].value_counts())
print("\nSummary of hours and ratings:")
clean[["hours_spent", "rating"]].describe()

### Q8. Compare ratings across events
`groupby` splits rows into groups so we can summarize each group separately.
For example, this computes median reported hours for each event:

```python
clean.groupby("event")["hours_spent"].median()
```

Compute the **count of non-missing ratings and mean rating** for each event using
`.agg(["count", "mean"])`. Sort the resulting table by `mean`, highest first.
Unlike the response counts above, this `count` excludes missing ratings.


In [ ]:
rating_summary = clean.groupby(...)[...].agg(["count", "mean"])
rating_summary = rating_summary.sort_values(..., ascending=False)
rating_summary

In [ ]:
grader.check("q8")

## 8. Tell the story

**Your response:** In 2–3 sentences:
- Which event has the highest average rating? Include its mean and number of valid ratings.
- Give one reason these responses alone cannot establish which event all students prefer.
  Consider the sample size, missing feedback, or who chose to respond.

> Write your answer here.

### Check your code
Run the next cell to recheck all six coding exercises. These checks give feedback
on code results; they do not assess written responses or submit your work.
If you change an earlier answer, rerun the cells after it before checking again.

In [ ]:
grader.check_all()